<a href="https://colab.research.google.com/github/GonzaloMartin/Python-Bootcamp/blob/main/Unidad_11/Python_Bootcamp_Clase_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/GonzaloMartin/Python-Bootcamp/refs/heads/main/Assets/python_bootcamp_banner1.png" width="400">

# **Python Bootcamp orientado a la Automatización**

Esta unidad cubre temas avanzados de pruebas de rendimiento para servicios backend.

# Unidad 11

En esta unidad explicaremos detalladamente los conceptos inherentes a pruebas de performance.

Al finalizar esta unidad deberían poder:

- Comprender qué son las pruebas de rendimiento.
- Diferenciar pruebas de Smoke, Load, Performance y Stress.
- Entender cómo funciona k6.
- Ejecutar una prueba básica.
- Comprender la estructura de un framework de performance.
- Interpretar métricas básicas.

***

## ¿Qué es una prueba de rendimiento?

Las pruebas de rendimiento permiten medir cómo responde un sistema bajo determinadas condiciones de carga.

A diferencia de una automatización funcional, donde validamos que algo funcione correctamente, en performance buscamos responder preguntas como:

- ¿Cuánto tarda en responder un sistema?
- ¿Cuántos usuarios soporta?
- ¿Cuántas requests por segundo puede manejar?
- ¿Se degrada el servicio bajo carga?
- ¿Cuándo empieza a fallar?
- ¿Qué ocurre si el tráfico aumenta repentinamente?

## ¿Qué es k6?

k6 es una herramienta open-source orientada a pruebas de carga y rendimiento.

Permite:

- Simular usuarios virtuales.
- Ejecutar requests HTTP.
- Medir tiempos de respuesta.
- Validar thresholds.
- Integrarse con CI/CD.
- Exportar métricas.
- Generar reportes.

Por lo general k6 usa JavaScript como lenguaje principal. Aunque puede usarse con otros lenguajes, pero lo recomendable es usar JavaScript.


## Primera prueba simple

### Archivo: `script.js`

Crear un archivo, dentro de una carpeta, que se llame `script.js` y ponerle este contenido:

```javascript
import http from 'k6/http';

export default function () {
    http.get('https://test-api.k6.io/');
}
```

### Ejecución

Ejecutarlo con el siguiente commando:

```bash
k6 run script.js
```

## Conceptos Fundamentales

### Virtual Users (VU)

Representan usuarios simulados.

Ejemplo:

```javascript
vus: 10
```

Significa:

- 10 usuarios concurrentes ejecutando el script.


### Duration

Tiempo total de ejecución.

```javascript
duration: '30s'
```

### Iteration

Cantidad de veces que se ejecuta el flujo.

### Estructura básica de una prueba

```javascript
import http from 'k6/http';

export const options = {
    vus: 10,
    duration: '30s',
};

export default function () {
    http.get('https://test-api.k6.io/');
}
```


## Qué métricas analiza k6

- **http_req_duration:** Tiempo total de respuesta.
- **http_req_failed:** Porcentaje de requests fallidas.
- **checks:** Validaciones exitosas/fallidas.
- **iterations:** Cantidad total de iteraciones.
- **http_reqs:** Cantidad total de requests ejecutadas.
- **Percentiles** En performance normalmente NO usamos promedios únicamente. Usamos percentiles. Por ejemplo, p(90), p(95), p(99).
- **Interpretación:** Por ejemplo, si un p95 tiene como resultado 450ms, significa que el 95% de las requests respondieron en menos de 450ms.
- **Thresholds:** Permiten definir reglas de aceptación. Superadas esas reglas, toda la prueba falla o se desestima.

### Ejemplo

```javascript
export const options = {
    thresholds: {
        http_req_failed: ['rate<0.01'],
        http_req_duration: ['p(95)<500'],
    },
};
```

## Checks

Son validaciones funcionales dentro de performance.
Una validación básica sobre la respuesta de un servicio suele ser su status code:

```javascript
import { check } from 'k6';

check(response, {
    'status es 200': (r) => r.status === 200,
});
```


## Tipos de pruebas de rendimiento

### Smoke Test

Validar que el servicio responde, no se cae y funciona correctamente con baja carga.
Se suele configurar con pocos usuarios, poco tiempo y es de ejecución rápida.

**Ejemplo:**

```text
5 usuarios durante 1 minuto
```

### Load Test

Se evalua el comportamiento bajo carga esperada, pero pesada.
Se sumla el tráfico real, además de buscar medir estabilidad, analiza la posible degradación del servicio.

**Ejemplo:**

```text
100 usuarios concurrentes
```

### Performance Test

Busca medir rendimiento sostenido prolongado pero con una carga menos pesada que una prueba de carga.
De duraciones más largas, se analizan percentiles y cambios en los umbrales de prueba.

**Ejemplo:**

```text
500 usuarios durante 30 minutos
```

### Stress Test

El único objetivo de una prueba de stress es llevar el sistema al límite.
Si una prueba de stress no resulta en error, entonces la configuración de la ejecución debe revisarse, porque todo sistema tiene un límite.
Se incrementa carga agresivamente. Se busca detectar puntos de quiebre. Y luego analizar la recuperación del sistema luego de la carga agresiva.

**Ejemplo:**

```text
>1000 usuarios durante 30 minutos, dependiendo del sistema.
```


## Diferencia entre los tipos

| Tipo | Objetivo |
|---|---|
| Smoke | Validar funcionamiento básico |
| Load | Evaluar carga esperada |
| Performance | Medir rendimiento sostenido |
| Stress | Romper el sistema |

## Ejemplo completo básico

```javascript
import http from 'k6/http';
import { check } from 'k6';

export const options = {
    vus: 10,
    duration: '30s',
    thresholds: {
        http_req_failed: ['rate<0.01'],
        http_req_duration: ['p(95)<500'],
    },
};

export default function () {

    const response = http.get('https://test-api.k6.io/');

    check(response, {
        'status es 200': (r) => r.status === 200,
    });
}
```


## Buenas prácticas

- No usar datos hardcodeados.
- Separar configuración.
- Versionar thresholds.
- Mantener scripts reutilizables.
- Evitar lógica compleja en tests.
- Centralizar helpers.
- Ejecutar desde CI/CD.
- Generar reportes automáticos.


## Errores comunes

- Medir solo promedio.
- No usar percentiles.
- Ejecutar desde una PC inestable.
- No controlar ambiente.
- Mezclar validaciones funcionales complejas.
- No separar perfiles.

